# pandas Interview Refresher

Practice selection, groupby, windows, reshaping, joins, strings, and dates with traceable outputs.

- **Study time:** 35-45 minutes
- **Prerequisites:** NumPy arrays and basic SQL-style grouping
- **Mode:** `quick`
- **Data policy:** no external files or downloads; a deterministic event table is created in memory
- **Provenance:** rebuilt from the curated pandas interview recap notebook

Output convention: every retained textual result begins with a label that identifies the operation that produced it.


In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)


def show(label, value):
    print(f"\n--- {label} ---\n{value}")


events = pd.DataFrame(
    {
        "event_id": np.arange(1, 13),
        "user_id": [101, 101, 102, 101, 103, 102, 103, 103, 101, 102, 103, 102],
        "timestamp": pd.to_datetime(
            [
                "2026-01-01 09:00",
                "2026-01-01 11:00",
                "2026-01-01 09:30",
                "2026-01-02 08:00",
                "2026-01-02 10:15",
                "2026-01-03 12:00",
                "2026-01-03 12:30",
                "2026-01-04 14:00",
                "2026-01-05 09:00",
                "2026-01-05 10:00",
                "2026-01-05 11:00",
                "2026-01-06 16:00",
            ]
        ),
        "channel": [
            "web",
            "app",
            "web",
            "store",
            "app",
            "web",
            "store",
            "app",
            "web",
            "store",
            "web",
            "app",
        ],
        "revenue": [20, 35, 15, 60, 10, 45, 55, 25, 80, 30, 50, 70],
        "note": [f"order_id=ORD-{value:03d}" for value in range(1, 13)],
    }
)
show("Source | event table", events.to_string(index=False))


--- Source | event table ---
 event_id  user_id           timestamp channel  revenue             note
        1      101 2026-01-01 09:00:00     web       20 order_id=ORD-001
        2      101 2026-01-01 11:00:00     app       35 order_id=ORD-002
        3      102 2026-01-01 09:30:00     web       15 order_id=ORD-003
        4      101 2026-01-02 08:00:00   store       60 order_id=ORD-004
        5      103 2026-01-02 10:15:00     app       10 order_id=ORD-005
        6      102 2026-01-03 12:00:00     web       45 order_id=ORD-006
        7      103 2026-01-03 12:30:00   store       55 order_id=ORD-007
        8      103 2026-01-04 14:00:00     app       25 order_id=ORD-008
        9      101 2026-01-05 09:00:00     web       80 order_id=ORD-009
       10      102 2026-01-05 10:00:00   store       30 order_id=ORD-010
       11      103 2026-01-05 11:00:00     web       50 order_id=ORD-011
       12      102 2026-01-06 16:00:00     app       70 order_id=ORD-012


## 1. `loc`, `iloc`, and safe assignment


In [2]:
high_value = events.loc[events["revenue"] >= 50, ["event_id", "user_id", "revenue"]]
positional = events.iloc[:3, :4]
labeled = events.copy()
labeled.loc[labeled["revenue"] >= 50, "value_band"] = "high"
labeled.loc[labeled["revenue"] < 50, "value_band"] = "regular"

show("Selection | loc revenue >= 50", high_value.to_string(index=False))
show("Selection | iloc first 3 rows and 4 columns", positional.to_string(index=False))
show("Assignment | value_band counts", labeled["value_band"].value_counts().to_string())


--- Selection | loc revenue >= 50 ---
 event_id  user_id  revenue
        4      101       60
        7      103       55
        9      101       80
       11      103       50
       12      102       70

--- Selection | iloc first 3 rows and 4 columns ---
 event_id  user_id           timestamp channel
        1      101 2026-01-01 09:00:00     web
        2      101 2026-01-01 11:00:00     app
        3      102 2026-01-01 09:30:00     web

--- Assignment | value_band counts ---
value_band
regular    7
high       5


## 2. Sorting, deduplication, and top-k per group


In [3]:
latest_per_user = (
    events.sort_values(["user_id", "timestamp", "event_id"])
    .drop_duplicates("user_id", keep="last")
    .sort_values("user_id")
)
top_two = (
    events.sort_values(["user_id", "revenue"], ascending=[True, False])
    .groupby("user_id", group_keys=False)
    .head(2)
)

show(
    "Dedup | latest event per user",
    latest_per_user[["user_id", "event_id", "timestamp"]].to_string(index=False),
)
show(
    "Ranking | top 2 revenue events per user",
    top_two[["user_id", "event_id", "revenue"]].to_string(index=False),
)


--- Dedup | latest event per user ---
 user_id  event_id           timestamp
     101         9 2026-01-05 09:00:00
     102        12 2026-01-06 16:00:00
     103        11 2026-01-05 11:00:00

--- Ranking | top 2 revenue events per user ---
 user_id  event_id  revenue
     101         9       80
     101         4       60
     102        12       70
     102         6       45
     103         7       55
     103        11       50


## 3. `agg` reduces rows; `transform` preserves rows


In [4]:
user_summary = events.groupby("user_id").agg(
    event_count=("event_id", "size"),
    total_revenue=("revenue", "sum"),
    mean_revenue=("revenue", "mean"),
)
with_group_features = events.assign(
    user_mean_revenue=events.groupby("user_id")["revenue"].transform("mean")
)
with_group_features["above_user_mean"] = (
    with_group_features["revenue"] > with_group_features["user_mean_revenue"]
)

show("Groupby agg | one row per user", user_summary.to_string())
show(
    "Groupby transform | row-aligned feature sample",
    with_group_features[["event_id", "user_id", "revenue", "user_mean_revenue", "above_user_mean"]]
    .head(8)
    .to_string(index=False),
)


--- Groupby agg | one row per user ---
         event_count  total_revenue  mean_revenue
user_id                                          
101                4            195         48.75
102                4            160         40.00
103                4            140         35.00

--- Groupby transform | row-aligned feature sample ---
 event_id  user_id  revenue  user_mean_revenue  above_user_mean
        1      101       20              48.75            False
        2      101       35              48.75            False
        3      102       15              40.00            False
        4      101       60              48.75             True
        5      103       10              35.00            False
        6      102       45              40.00             True
        7      103       55              35.00             True
        8      103       25              35.00            False


## 4. Time ordering, shift, gaps, and lagged rolling features


In [5]:
ordered = events.sort_values(["user_id", "timestamp", "event_id"]).copy()
ordered["previous_timestamp"] = ordered.groupby("user_id")["timestamp"].shift(1)
ordered["gap_hours"] = (
    ordered["timestamp"] - ordered["previous_timestamp"]
).dt.total_seconds() / 3600
ordered["previous_revenue"] = ordered.groupby("user_id")["revenue"].shift(1)
ordered["prior_two_mean"] = (
    ordered.groupby("user_id")["previous_revenue"]
    .rolling(2, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)
ordered["month"] = ordered["timestamp"].dt.to_period("M")

show(
    "Time features | previous event, gap, lagged rolling mean",
    ordered[["user_id", "timestamp", "revenue", "gap_hours", "prior_two_mean"]].to_string(
        index=False
    ),
)


--- Time features | previous event, gap, lagged rolling mean ---
 user_id           timestamp  revenue  gap_hours  prior_two_mean
     101 2026-01-01 09:00:00       20        NaN             NaN
     101 2026-01-01 11:00:00       35       2.00            20.0
     101 2026-01-02 08:00:00       60      21.00            27.5
     101 2026-01-05 09:00:00       80      73.00            47.5
     102 2026-01-01 09:30:00       15        NaN             NaN
     102 2026-01-03 12:00:00       45      50.50            15.0
     102 2026-01-05 10:00:00       30      46.00            30.0
     102 2026-01-06 16:00:00       70      30.00            37.5
     103 2026-01-02 10:15:00       10        NaN             NaN
     103 2026-01-03 12:30:00       55      26.25            10.0
     103 2026-01-04 14:00:00       25      25.50            32.5
     103 2026-01-05 11:00:00       50      21.00            40.0


## 5. Strings and categorical cleanup


In [6]:
string_features = events[["event_id", "note", "channel"]].copy()
string_features["order_id"] = string_features["note"].str.extract(r"(ORD-\d+)")
string_features["channel"] = string_features["channel"].astype("category")

show("Strings | extracted order identifiers", string_features.head(6).to_string(index=False))
show("Categorical | channel categories", string_features["channel"].cat.categories.tolist())


--- Strings | extracted order identifiers ---
 event_id             note channel order_id
        1 order_id=ORD-001     web  ORD-001
        2 order_id=ORD-002     app  ORD-002
        3 order_id=ORD-003     web  ORD-003
        4 order_id=ORD-004   store  ORD-004
        5 order_id=ORD-005     app  ORD-005
        6 order_id=ORD-006     web  ORD-006

--- Categorical | channel categories ---
['app', 'store', 'web']


## 6. Pivot, pivot table, and melt


In [7]:
revenue_matrix = events.pivot_table(
    index="user_id",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
)
revenue_matrix.columns = [f"revenue_{column}" for column in revenue_matrix.columns]
wide = revenue_matrix.reset_index()
long = wide.melt(id_vars="user_id", var_name="metric", value_name="value")

show("Reshape | revenue pivot table", wide.to_string(index=False))
show("Reshape | melted long form", long.head(9).to_string(index=False))


--- Reshape | revenue pivot table ---
 user_id  revenue_app  revenue_store  revenue_web
     101           35             60          100
     102           70             30           60
     103           35             55           50

--- Reshape | melted long form ---
 user_id        metric  value
     101   revenue_app     35
     102   revenue_app     70
     103   revenue_app     35
     101 revenue_store     60
     102 revenue_store     30
     103 revenue_store     55
     101   revenue_web    100
     102   revenue_web     60
     103   revenue_web     50


## 7. Validated joins and unmatched-key checks


In [8]:
users = pd.DataFrame(
    {
        "user_id": [101, 102, 103, 104],
        "segment": ["growth", "core", "growth", "new"],
    }
)
joined = events.merge(users, on="user_id", how="left", validate="many_to_one", indicator=True)
unmatched = joined.loc[joined["_merge"] != "both", ["event_id", "user_id", "_merge"]]

show("Join | events enriched with segment", joined.head(8).to_string(index=False))
show(
    "Join audit | unmatched event keys",
    unmatched.to_string(index=False) if len(unmatched) else "none",
)


--- Join | events enriched with segment ---
 event_id  user_id           timestamp channel  revenue             note segment _merge
        1      101 2026-01-01 09:00:00     web       20 order_id=ORD-001  growth   both
        2      101 2026-01-01 11:00:00     app       35 order_id=ORD-002  growth   both
        3      102 2026-01-01 09:30:00     web       15 order_id=ORD-003    core   both
        4      101 2026-01-02 08:00:00   store       60 order_id=ORD-004  growth   both
        5      103 2026-01-02 10:15:00     app       10 order_id=ORD-005  growth   both
        6      102 2026-01-03 12:00:00     web       45 order_id=ORD-006    core   both
        7      103 2026-01-03 12:30:00   store       55 order_id=ORD-007  growth   both
        8      103 2026-01-04 14:00:00     app       25 order_id=ORD-008  growth   both

--- Join audit | unmatched event keys ---
none


## 8. Retrieval checks


In [9]:
assert top_two.groupby("user_id").size().eq(2).all()
assert joined["event_id"].is_unique
assert long.shape[0] == len(wide) * len(revenue_matrix.columns)
assert (
    ordered.groupby("user_id")["timestamp"]
    .apply(lambda values: values.is_monotonic_increasing)
    .all()
)

show("Drill checks | status", "all assertions passed")


--- Drill checks | status ---
all assertions passed
